<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/BENCH_TOPO_COMPLETE_FULLHOPE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://zenodo.org/records/21245474

https://www.thinkers360.com/tl/profiles/view/25153


## SETUP

In [2]:
!pip install bitsandbytes -q

In [3]:
!git clone https://github.com/frank-morales2020/ast_lefm.git
%cd /content/ast_lefm
!pip install -e .

fatal: destination path 'ast_lefm' already exists and is not an empty directory.
/content/ast_lefm
Obtaining file:///content/ast_lefm
  Preparing metadata (setup.py) ... done
  Attempting uninstall: ast_lefm
    Found existing installation: ast_lefm 1.0.0
    Uninstalling ast_lefm-1.0.0:
      Successfully uninstalled ast_lefm-1.0.0
  Running setup.py develop for ast_lefm


In [4]:
import sys
import os

# Force add the path
sys.path.insert(0, '/content/ast_lefm')
sys.path.insert(0, '/content/ast_lefm/ast_lefm')

# Try to import
try:
    from ast_lefm.sieve import primes_up_to
    print("✓ Import successful")
except ImportError as e:
    print(f"✗ Import failed: {e}")
    # Let's see what's in the directory
    print("\nFiles in /content/ast_lefm:")
    os.listdir('/content/ast_lefm')

    print("\nTrying alternative import...")
    # Try importing as a local module
    import importlib.util
    spec = importlib.util.spec_from_file_location("sieve", "/content/ast_lefm/sieve.py")
    sieve = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(sieve)
    primes_up_to = sieve.primes_up_to
    print("✓ Alternative import worked")

✓ Import successful


## FULL HOPE - 500

In [1]:
# =====================================================================
# COMPLETE BENCHMARK - BASELINE | EWC | REPLAY | FULL HOPE | TOPOLOGICAL
# CORRECTED: proj_gate dtype fix (bfloat16)
# =====================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import gc
import bitsandbytes as bnb
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from sklearn.metrics import accuracy_score
from warnings import filterwarnings
from typing import Dict, Tuple, Optional, List
filterwarnings('ignore')

SEED = 123
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "openai/gpt-oss-20b"
HIDDEN_SIZE = 2880
PRIME_LIMIT = 13

# =====================================================================
# CONFIGURATION
# =====================================================================
EPOCHS = 3
SAMPLES = 500
BATCH_SIZE = 16
NUM_RUNS = 5

LR_GRID = [
    (5e-3, 1e-3),   # Run 0
    (1e-3, 5e-4),   # Run 1
    (1e-2, 2e-3),   # Run 2
    (5e-3, 5e-3),   # Run 3
    (2e-3, 1e-3),   # Run 4
]

print(f"Device: {device}")
print("=" * 80)
print("BENCHMARK: BASELINE | EWC | REPLAY | FULL HOPE | TOPOLOGICAL")
print("=" * 80)


# =====================================================================
# PART 1: FULL HOPE ARCHITECTURE (CORRECTED - proj_gate dtype fix)
# =====================================================================

class AssociativeMemoryModule(nn.Module):
    """
    Associative memory: retrieves relevant past experiences based on similarity.
    512 memory slots with novelty detection via cosine similarity.
    FIXED: proj_gate created with bfloat16 dtype for compatibility
    """
    def __init__(self, hidden_dim: int, memory_slots: int = 512):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.memory_slots = memory_slots
        self.register_buffer("memory_keys", torch.randn(memory_slots, hidden_dim) * 0.02)
        self.register_buffer("memory_values", torch.randn(memory_slots, hidden_dim) * 0.02)

        # FIX: Create proj_gate with bfloat16 dtype to match hidden states
        self.proj_gate = nn.Linear(hidden_dim * 2, 1, dtype=torch.bfloat16)

        self.register_buffer("access_count", torch.zeros(memory_slots))

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Retrieval + gating mechanism.
        FIX: Cast memory buffers to match input dtype (bfloat16)
        Returns: (output with integrated memory, attention weights)
        """
        norm_x = F.normalize(x, dim=-1)

        # Cast memory buffers to match input dtype (e.g., bfloat16)
        memory_keys = self.memory_keys.to(x.dtype)
        memory_values = self.memory_values.to(x.dtype)

        norm_keys = F.normalize(memory_keys, dim=-1)
        scores = torch.matmul(norm_x, norm_keys.T)
        weights = F.softmax(scores / 0.1, dim=-1)
        retrieved = torch.matmul(weights, memory_values)
        gate = torch.sigmoid(self.proj_gate(torch.cat([x, retrieved], dim=-1)))
        output = gate * x + (1 - gate) * retrieved

        with torch.no_grad():
            slot_usage = weights.mean(dim=(0, 1))
            self.access_count += slot_usage

        return output, weights


class FullHOPEModel(nn.Module):
    """
    HOPE (Hierarchically Organized Predictive Experts):
    - Runs entire base transformer (which handles position embeddings internally)
    - Applies associative memory retrieval + gating on output
    - Classifies based on pooled memory-integrated representations

    KEY INSIGHT: Don't wrap individual transformer layers.
    They need position_embeddings computed internally by base_model.
    Instead, apply HOPE's associative memory on top of the transformer's output.
    """
    def __init__(self, base_transformer: nn.Module, hidden_dim: int = HIDDEN_SIZE):
        super().__init__()
        self.base_transformer = base_transformer
        self.hidden_dim = hidden_dim
        self.associative_memory = AssociativeMemoryModule(hidden_dim)
        self.classifier = nn.Linear(hidden_dim, 2, dtype=torch.bfloat16)
        print(f"  [HOPE] Initialized with associative memory (512 slots)")

    def forward(self, input_ids: torch.Tensor, attention_mask: Optional[torch.Tensor] = None,
                update_fast: bool = True) -> torch.Tensor:
        """
        Forward pass:
        1. Run entire transformer through base model (handles position embeddings)
        2. Apply associative memory retrieval + gating
        3. Pool and classify
        """
        # STEP 1: Run entire transformer
        outputs = self.base_transformer(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        hidden_states = outputs.hidden_states[-1]  # [batch, seq, hidden]

        # STEP 2: Integrate associative memory
        hidden_states, _ = self.associative_memory(hidden_states)

        # STEP 3: Pool by attending to last valid token
        if attention_mask is not None:
            seq_lens = torch.eq(attention_mask, 1).int().sum(-1) - 1
            batch_idx = torch.arange(input_ids.shape[0], device=input_ids.device)
            pooled = hidden_states[batch_idx, seq_lens, :]
        else:
            pooled = hidden_states[:, -1, :]

        # STEP 4: Classify
        return self.classifier(pooled)

    def trigger_nested_consolidation(self):
        """Placeholder for consolidation (future: fast/medium/slow weight consolidation)"""
        pass


# =====================================================================
# PART 2: DATA LOADING
# =====================================================================

dataset = load_dataset('SetFit/ag_news', split='train')

def create_task(dataset, class_labels, num_samples=SAMPLES):
    filtered = dataset.filter(lambda x: x['label'] in class_labels)
    sampled = filtered.select(range(min(num_samples, len(filtered))))
    texts = [item['text'] for item in sampled]
    labels = [item['label'] % 2 for item in sampled]
    return texts, labels

task_a_texts, task_a_labels = create_task(dataset, [0, 1], SAMPLES)
task_b_texts, task_b_labels = create_task(dataset, [2, 3], SAMPLES)

print(f"Task A: {len(task_a_texts)} samples (World vs Sports)")
print(f"Task B: {len(task_b_texts)} samples (Business vs Sci/Tech)")


# =====================================================================
# PART 3: SHARED CLASSIFIER MODEL
# =====================================================================

class SharedClassifierModel(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model
        self.classifier = nn.Linear(HIDDEN_SIZE, 2, dtype=torch.bfloat16)

    def forward(self, input_ids, attention_mask=None):
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        hidden_states = outputs.hidden_states[-1]

        if attention_mask is not None:
            seq_lens = torch.eq(attention_mask, 1).int().sum(-1) - 1
            batch_idx = torch.arange(input_ids.shape[0], device=input_ids.device)
            last_hidden = hidden_states[batch_idx, seq_lens, :]
        else:
            last_hidden = hidden_states[:, -1, :]

        return self.classifier(last_hidden)


# =====================================================================
# PART 4: TOPOLOGICAL MODEL
# =====================================================================

class TopologicalModel(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model
        dev = next(base_model.parameters()).device
        self.classifier_A = nn.Linear(HIDDEN_SIZE, 2, dtype=torch.bfloat16).to(dev)
        self.classifier_B = nn.Linear(HIDDEN_SIZE, 2, dtype=torch.bfloat16).to(dev)
        self.current_task = 'A'

    def forward(self, input_ids, attention_mask=None):
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        hidden_states = outputs.hidden_states[-1]

        if attention_mask is not None:
            seq_lens = torch.eq(attention_mask, 1).int().sum(-1) - 1
            batch_idx = torch.arange(input_ids.shape[0], device=input_ids.device)
            last_hidden = hidden_states[batch_idx, seq_lens, :]
        else:
            last_hidden = hidden_states[:, -1, :]

        head = getattr(self, f'classifier_{self.current_task}')
        return head(last_hidden)

    def switch_task(self, task: str):
        assert task in ('A', 'B')
        self.current_task = task

    def freeze_previous_head(self):
        self.classifier_A.requires_grad_(False)


# =====================================================================
# PART 5: TOPOLOGICAL GOVERNOR
# =====================================================================

class TopologicalGovernor:
    def __init__(self, embed_layer: nn.Embedding, prime_limit: int = PRIME_LIMIT):
        self.embed_layer = embed_layer
        vocab_size = embed_layer.weight.shape[0]

        sieve = [True] * (prime_limit + 1)
        sieve[0] = sieve[1] = False
        for i in range(2, int(prime_limit ** 0.5) + 1):
            if sieve[i]:
                for j in range(i * i, prime_limit + 1, i):
                    sieve[j] = False
        primes = [i for i in range(2, prime_limit + 1) if sieve[i]]
        self.anchor_indices = [p for p in primes if p < vocab_size]
        self.snapshot = {}
        self.safety_constant = 1.0 - np.prod([1.0 - (p ** -0.5) for p in self.anchor_indices])

        print(f"  [GOVERNOR] Anchoring {len(self.anchor_indices)} prime coords: {self.anchor_indices}")
        print(f"  [GOVERNOR] Safety Constant Λ: {self.safety_constant:.10f}")

    def take_snapshot(self):
        self.snapshot = {
            idx: self.embed_layer.weight[idx].detach().clone().float()
            for idx in self.anchor_indices
        }

    @torch.no_grad()
    def zero_anchor_gradients(self):
        if self.embed_layer.weight.grad is not None:
            for idx in self.anchor_indices:
                self.embed_layer.weight.grad[idx].zero_()

    @torch.no_grad()
    def enforce_anchors(self):
        if not self.snapshot:
            return
        dtype = self.embed_layer.weight.dtype
        for idx, cached in self.snapshot.items():
            self.embed_layer.weight[idx].copy_(cached.to(dtype=dtype))

    def verify_integrity(self, atol: float = 1e-5) -> bool:
        if not self.snapshot:
            return True
        return all(
            torch.allclose(self.embed_layer.weight[idx].float(), cached, atol=atol)
            for idx, cached in self.snapshot.items()
        )


# =====================================================================
# PART 6: EWC
# =====================================================================

class EWC:
    def __init__(self, model, fisher_samples=200):
        self.model = model
        self.fisher_samples = fisher_samples
        self.params = {n: p for n, p in model.named_parameters() if p.requires_grad}
        self._means = {}
        self._precision_matrices = {}

    def update_fisher(self, dataloader):
        for n, p in self.params.items():
            self._precision_matrices[n] = torch.zeros_like(p).to(device)

        self.model.eval()
        for batch in dataloader:
            self.model.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            logits = self.model(input_ids, attention_mask)
            loss = F.cross_entropy(logits, labels)
            loss.backward()

            for n, p in self.params.items():
                if p.grad is not None:
                    self._precision_matrices[n] += p.grad ** 2 / len(dataloader)

        self.model.train()

    def update_means(self):
        for n, p in self.params.items():
            self._means[n] = p.clone().detach()

    def penalty(self, lambda_ewc=5000):
        loss = 0
        for n, p in self.params.items():
            if n in self._precision_matrices:
                _loss = self._precision_matrices[n] * (p - self._means[n]) ** 2
                loss += _loss.sum()
        return lambda_ewc * loss


# =====================================================================
# PART 7: EXPERIENCE REPLAY
# =====================================================================

class ExperienceReplay:
    def __init__(self, buffer_size=200):
        self.buffer_size = buffer_size
        self.buffer = []

    def add(self, texts, labels):
        self.buffer.extend(list(zip(texts, labels)))
        if len(self.buffer) > self.buffer_size:
            self.buffer = self.buffer[-self.buffer_size:]

    def sample(self, batch_size=16):
        if len(self.buffer) < batch_size:
            return [], []
        indices = np.random.choice(len(self.buffer), batch_size, replace=False)
        texts, labels = zip(*[self.buffer[i] for i in indices])
        return list(texts), list(labels)


# =====================================================================
# PART 8: EVALUATION
# =====================================================================

def evaluate_model(model, tokenizer, texts, labels, batch_size=16):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i+batch_size]
            batch_labels = labels[i:i+batch_size]
            tokens = tokenizer(batch_texts, return_tensors="pt", padding=True,
                             truncation=True, max_length=64).to(device)

            # Check if model is Full HOPE
            if hasattr(model, 'associative_memory'):
                logits = model(tokens.input_ids, tokens.attention_mask, update_fast=False)
            else:
                logits = model(tokens.input_ids, tokens.attention_mask)

            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(batch_labels)

    return accuracy_score(all_labels, all_preds)


# =====================================================================
# PART 9: TRAINING FUNCTION
# =====================================================================

def train_model(method='baseline'):
    print(f"\n{'='*60}")
    print(f"Method: {method.upper()}")
    print(f"{'='*60}")

    all_results = []

    for run_id in range(NUM_RUNS):
        lr_embed, lr_cls = LR_GRID[run_id]
        print(f"\n  Run {run_id+1}/{NUM_RUNS} | lr_embed={lr_embed:.0e}, lr_cls={lr_cls:.0e}")

        base_model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME, trust_remote_code=True, torch_dtype=torch.bfloat16
        ).to(device)

        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
        tokenizer.pad_token = tokenizer.eos_token

        for param in base_model.parameters():
            param.requires_grad = False

        # Initialize variables before model creation
        ewc = None
        replay = None
        governor = None
        embed_layer = None

        # Create model
        if method == 'topological':
            model = TopologicalModel(base_model).to(device)
        elif method == 'full_hope':
            model = FullHOPEModel(base_model).to(device)
        else:
            model = SharedClassifierModel(base_model).to(device)

        # Setup optimizer
        if method == 'topological':
            if hasattr(base_model, "transformer") and hasattr(base_model.transformer, "wte"):
                embed_layer = base_model.transformer.wte
            else:
                for module in base_model.modules():
                    if hasattr(module, "weight") and module.weight.shape[0] > 100000:
                        embed_layer = module
                        break
            embed_layer.weight.requires_grad = True
            governor = TopologicalGovernor(embed_layer)

            optimizer = bnb.optim.AdamW8bit([
                {'params': embed_layer.weight, 'lr': lr_embed},
                {'params': model.classifier_A.parameters(), 'lr': lr_cls}
            ])
        elif method == 'full_hope':
            model.classifier.requires_grad_(True)
            optimizer = bnb.optim.AdamW8bit(model.classifier.parameters(), lr=lr_cls)
        else:
            model.classifier.requires_grad_(True)
            optimizer = bnb.optim.AdamW8bit(model.classifier.parameters(), lr=lr_cls)

        # ===== TASK A TRAINING =====
        if method == 'topological':
            model.switch_task('A')
        model.train()

        for epoch in range(EPOCHS):
            for i in range(0, len(task_a_texts), BATCH_SIZE):
                batch_texts = task_a_texts[i:i+BATCH_SIZE]
                batch_labels = torch.tensor(task_a_labels[i:i+BATCH_SIZE]).to(device)

                tokens = tokenizer(batch_texts, return_tensors="pt", padding=True,
                                 truncation=True, max_length=64).to(device)
                optimizer.zero_grad()

                if method == 'full_hope':
                    logits = model(tokens.input_ids, tokens.attention_mask, update_fast=True)
                else:
                    logits = model(tokens.input_ids, tokens.attention_mask)

                loss = F.cross_entropy(logits, batch_labels)
                loss.backward()

                if method == 'topological' and governor:
                    governor.zero_anchor_gradients()

                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

        # ===== POST-TASK A =====
        if method == 'topological' and governor:
            governor.take_snapshot()
            model.freeze_previous_head()
            optimizer = bnb.optim.AdamW8bit([
                {'params': embed_layer.weight, 'lr': lr_embed},
                {'params': model.classifier_B.parameters(), 'lr': lr_cls}
            ])

        elif method == 'ewc':
            ewc = EWC(model)
            dummy_dataloader = []
            for i in range(0, len(task_a_texts), 32):
                batch_texts = task_a_texts[i:i+32]
                batch_labels = torch.tensor(task_a_labels[i:i+32]).to(device)
                tokens = tokenizer(batch_texts, return_tensors="pt", padding=True,
                                 truncation=True, max_length=64).to(device)
                dummy_dataloader.append({
                    'input_ids': tokens.input_ids,
                    'attention_mask': tokens.attention_mask,
                    'labels': batch_labels
                })
            ewc.update_fisher(dummy_dataloader)
            ewc.update_means()

        elif method == 'replay':
            replay = ExperienceReplay(buffer_size=200)
            replay.add(task_a_texts, task_a_labels)

        # ===== EVALUATE TASK A =====
        acc_a_initial = evaluate_model(model, tokenizer, task_a_texts[:200], task_a_labels[:200])
        print(f"      Task A Initial: {acc_a_initial*100:.1f}%")

        # ===== TASK B TRAINING =====
        if method == 'topological':
            model.switch_task('B')
        model.train()

        for epoch in range(EPOCHS):
            for i in range(0, len(task_b_texts), BATCH_SIZE):
                batch_texts = task_b_texts[i:i+BATCH_SIZE]
                batch_labels = torch.tensor(task_b_labels[i:i+BATCH_SIZE]).to(device)

                if method == 'replay' and replay:
                    replay_texts, replay_labels = replay.sample(batch_size=8)
                    if replay_texts:
                        batch_texts = batch_texts[:8] + replay_texts
                        batch_labels = torch.cat([
                            batch_labels[:8],
                            torch.tensor(replay_labels).to(device)
                        ])

                tokens = tokenizer(batch_texts, return_tensors="pt", padding=True,
                                 truncation=True, max_length=64).to(device)
                optimizer.zero_grad()

                if method == 'full_hope':
                    logits = model(tokens.input_ids, tokens.attention_mask, update_fast=True)
                else:
                    logits = model(tokens.input_ids, tokens.attention_mask)

                loss = F.cross_entropy(logits, batch_labels)

                if method == 'ewc' and ewc:
                    loss += ewc.penalty(lambda_ewc=500)

                loss.backward()

                if method == 'topological' and governor:
                    governor.zero_anchor_gradients()

                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

                if method == 'topological' and governor:
                    governor.enforce_anchors()

                if method == 'full_hope':
                    if i % 50 == 0:
                        model.trigger_nested_consolidation()

        # ===== FINAL EVALUATION =====
        if method == 'topological':
            model.switch_task('A')
            acc_a_final = evaluate_model(model, tokenizer, task_a_texts[:200], task_a_labels[:200])
            model.switch_task('B')
            acc_b = evaluate_model(model, tokenizer, task_b_texts[:200], task_b_labels[:200])
        else:
            acc_a_final = evaluate_model(model, tokenizer, task_a_texts[:200], task_a_labels[:200])
            acc_b = evaluate_model(model, tokenizer, task_b_texts[:200], task_b_labels[:200])

        forgetting = (acc_a_initial - acc_a_final) * 100

        print(f"      Task A Final: {acc_a_final*100:.1f}%")
        print(f"      Task B Acc: {acc_b*100:.1f}%")
        print(f"      Forgetting: {forgetting:.1f}%")

        if method == 'topological' and governor:
            if governor.verify_integrity():
                print(f"      ✅ Anchors intact")

        all_results.append({
            'run_id': run_id,
            'lr_embed': lr_embed,
            'lr_cls': lr_cls,
            'acc_a_initial': acc_a_initial,
            'acc_a_final': acc_a_final,
            'forgetting': forgetting,
            'acc_b': acc_b
        })

        del model, base_model, tokenizer, optimizer
        if method == 'topological' and governor is not None:
            del governor
        elif method == 'ewc' and ewc is not None:
            del ewc
        elif method == 'replay' and replay is not None:
            del replay

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

    return {
        'results': all_results,
        'best_run': max(all_results, key=lambda x: x['acc_b']),
        'mean_forgetting': np.mean([r['forgetting'] for r in all_results]),
        'mean_acc_b': np.mean([r['acc_b'] for r in all_results]),
        'std_forgetting': np.std([r['forgetting'] for r in all_results]),
        'std_acc_b': np.std([r['acc_b'] for r in all_results]),
    }


# =====================================================================
# PART 10: RUN BENCHMARK
# =====================================================================
print("\n" + "=" * 80)
print("RUNNING BENCHMARK")
print("=" * 80)

methods = ['baseline', 'ewc', 'replay', 'full_hope', 'topological']
results = {}

for method in methods:
    results[method] = train_model(method=method)


# =====================================================================
# PART 11: RESULTS TABLE
# =====================================================================
print("\n" + "=" * 80)
print("FINAL RESULTS")
print("=" * 80)
print(f"{'Method':<15} {'Best Fgt':<12} {'Mean Fgt':<12} {'Best Task B':<12} {'Mean Task B':<12}")
print("-" * 70)

for method in methods:
    r = results[method]
    best = r['best_run']
    print(f"{method.upper():<15} {best['forgetting']:.1f}%{'':<6} "
          f"{r['mean_forgetting']:.1f}%{'':<6} "
          f"{best['acc_b']*100:.1f}%{'':<6} "
          f"{r['mean_acc_b']*100:.1f}%")

print("-" * 70)

# Ranking
print("\n" + "=" * 80)
print("RANKING (Best to Worst by Mean Forgetting):")
print("=" * 80)

sorted_methods = sorted(methods, key=lambda m: results[m]['mean_forgetting'])
for i, method in enumerate(sorted_methods, 1):
    r = results[method]
    print(f"{i}. {method.upper()}: {r['mean_forgetting']:.1f}% forgetting "
          f"(Best Task B: {r['best_run']['acc_b']*100:.1f}%)")

print("\n" + "=" * 80)
print("VERDICT:")
print("=" * 80)

topo_best = results['topological']['best_run']
hope_best = results['full_hope']['best_run']
replay_best = results['replay']['best_run']

print(f"🔵 TOPOLOGICAL AI:")
print(f"   • Forgetting: {topo_best['forgetting']:.1f}%")
print(f"   • Task B Accuracy: {topo_best['acc_b']*100:.1f}%")

print(f"\n🟣 FULL HOPE:")
print(f"   • Forgetting: {hope_best['forgetting']:.1f}%")
print(f"   • Task B Accuracy: {hope_best['acc_b']*100:.1f}%")

print(f"\n🟢 EXPERIENCE REPLAY:")
print(f"   • Forgetting: {replay_best['forgetting']:.1f}%")
print(f"   • Task B Accuracy: {replay_best['acc_b']*100:.1f}%")

if topo_best['forgetting'] < hope_best['forgetting']:
    improvement = hope_best['forgetting'] - topo_best['forgetting']
    print(f"\n✅ TOPOLOGICAL AI WINS!")
    print(f"   • {improvement:.1f}% less forgetting than Full HOPE")
elif hope_best['forgetting'] < topo_best['forgetting']:
    improvement = topo_best['forgetting'] - hope_best['forgetting']
    print(f"\n✅ FULL HOPE WINS!")
    print(f"   • {improvement:.1f}% less forgetting than Topological AI")
else:
    print(f"\n⚖️ TIE!")

print("=" * 80)

Device: cuda
BENCHMARK: BASELINE | EWC | REPLAY | FULL HOPE | TOPOLOGICAL


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Task A: 500 samples (World vs Sports)
Task B: 500 samples (Business vs Sci/Tech)

RUNNING BENCHMARK

Method: BASELINE

  Run 1/5 | lr_embed=5e-03, lr_cls=1e-03


[transformers] MXFP4 quantization requires the `kernels` package: `pip install kernels>=0.12.0`. We will default to dequantizing the model to bf16.


Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

...

  [GOVERNOR] Anchoring 6 prime coords: [2, 3, 5, 7, 11, 13]
  [GOVERNOR] Safety Constant Λ: 0.9785142874
      Task A Initial: 96.0%
      Task A Final: 94.5%
      Task B Acc: 77.0%
      Forgetting: 1.5%
      ✅ Anchors intact

  Run 4/5 | lr_embed=5e-03, lr_cls=5e-03


Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

  [GOVERNOR] Anchoring 6 prime coords: [2, 3, 5, 7, 11, 13]
  [GOVERNOR] Safety Constant Λ: 0.9785142874
      Task A Initial: 96.5%
      Task A Final: 96.5%
      Task B Acc: 92.0%
      Forgetting: 0.0%
      ✅ Anchors intact

  Run 5/5 | lr_embed=2e-03, lr_cls=1e-03


Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

  [GOVERNOR] Anchoring 6 prime coords: [2, 3, 5, 7, 11, 13]
  [GOVERNOR] Safety Constant Λ: 0.9785142874
      Task A Initial: 99.5%
      Task A Final: 99.0%
      Task B Acc: 75.5%
      Forgetting: 0.5%
      ✅ Anchors intact

FINAL RESULTS
Method          Best Fgt     Mean Fgt     Best Task B  Mean Task B 
----------------------------------------------------------------------
BASELINE        44.0%       47.0%       67.0%       63.3%
EWC             38.5%       27.7%       64.5%       58.2%
REPLAY          13.5%       4.0%       79.0%       72.3%
FULL_HOPE       42.5%       45.4%       62.5%       61.5%
TOPOLOGICAL     0.0%       0.6%       92.0%       78.0%
----------------------------------------------------------------------

RANKING (Best to Worst by Mean Forgetting):
1. TOPOLOGICAL: 0.6% forgetting (Best Task B: 92.0%)
2. REPLAY: 4.0% forgetting (Best Task B: 79.0%)
3. EWC: 27.7% forgetting (Best Task B: 64.5%)
4. FULL_HOPE: 45.4% forgetting (Best Task B: 62.5%)
5. BASELINE: 47

## FULL HOPE - 1500

In [1]:
# =====================================================================
# COMPLETE BENCHMARK - BASELINE | EWC | REPLAY | FULL HOPE | TOPOLOGICAL
# CORRECTED: proj_gate dtype fix (bfloat16)
# =====================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import gc
import bitsandbytes as bnb
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from sklearn.metrics import accuracy_score
from warnings import filterwarnings
from typing import Dict, Tuple, Optional, List
filterwarnings('ignore')

SEED = 123
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "openai/gpt-oss-20b"
HIDDEN_SIZE = 2880
PRIME_LIMIT = 13

# =====================================================================
# CONFIGURATION
# =====================================================================
EPOCHS = 3
SAMPLES = 1500
BATCH_SIZE = 16
NUM_RUNS = 5

LR_GRID = [
    (5e-3, 1e-3),   # Run 0
    (1e-3, 5e-4),   # Run 1
    (1e-2, 2e-3),   # Run 2
    (5e-3, 5e-3),   # Run 3
    (2e-3, 1e-3),   # Run 4
]

print(f"Device: {device}")
print("=" * 80)
print("BENCHMARK: BASELINE | EWC | REPLAY | FULL HOPE | TOPOLOGICAL")
print("=" * 80)


# =====================================================================
# PART 1: FULL HOPE ARCHITECTURE (CORRECTED - proj_gate dtype fix)
# =====================================================================

class AssociativeMemoryModule(nn.Module):
    """
    Associative memory: retrieves relevant past experiences based on similarity.
    512 memory slots with novelty detection via cosine similarity.
    FIXED: proj_gate created with bfloat16 dtype for compatibility
    """
    def __init__(self, hidden_dim: int, memory_slots: int = 512):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.memory_slots = memory_slots
        self.register_buffer("memory_keys", torch.randn(memory_slots, hidden_dim) * 0.02)
        self.register_buffer("memory_values", torch.randn(memory_slots, hidden_dim) * 0.02)

        # FIX: Create proj_gate with bfloat16 dtype to match hidden states
        self.proj_gate = nn.Linear(hidden_dim * 2, 1, dtype=torch.bfloat16)

        self.register_buffer("access_count", torch.zeros(memory_slots))

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Retrieval + gating mechanism.
        FIX: Cast memory buffers to match input dtype (bfloat16)
        Returns: (output with integrated memory, attention weights)
        """
        norm_x = F.normalize(x, dim=-1)

        # Cast memory buffers to match input dtype (e.g., bfloat16)
        memory_keys = self.memory_keys.to(x.dtype)
        memory_values = self.memory_values.to(x.dtype)

        norm_keys = F.normalize(memory_keys, dim=-1)
        scores = torch.matmul(norm_x, norm_keys.T)
        weights = F.softmax(scores / 0.1, dim=-1)
        retrieved = torch.matmul(weights, memory_values)
        gate = torch.sigmoid(self.proj_gate(torch.cat([x, retrieved], dim=-1)))
        output = gate * x + (1 - gate) * retrieved

        with torch.no_grad():
            slot_usage = weights.mean(dim=(0, 1))
            self.access_count += slot_usage

        return output, weights


class FullHOPEModel(nn.Module):
    """
    HOPE (Hierarchically Organized Predictive Experts):
    - Runs entire base transformer (which handles position embeddings internally)
    - Applies associative memory retrieval + gating on output
    - Classifies based on pooled memory-integrated representations

    KEY INSIGHT: Don't wrap individual transformer layers.
    They need position_embeddings computed internally by base_model.
    Instead, apply HOPE's associative memory on top of the transformer's output.
    """
    def __init__(self, base_transformer: nn.Module, hidden_dim: int = HIDDEN_SIZE):
        super().__init__()
        self.base_transformer = base_transformer
        self.hidden_dim = hidden_dim
        self.associative_memory = AssociativeMemoryModule(hidden_dim)
        self.classifier = nn.Linear(hidden_dim, 2, dtype=torch.bfloat16)
        print(f"  [HOPE] Initialized with associative memory (512 slots)")

    def forward(self, input_ids: torch.Tensor, attention_mask: Optional[torch.Tensor] = None,
                update_fast: bool = True) -> torch.Tensor:
        """
        Forward pass:
        1. Run entire transformer through base model (handles position embeddings)
        2. Apply associative memory retrieval + gating
        3. Pool and classify
        """
        # STEP 1: Run entire transformer
        outputs = self.base_transformer(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        hidden_states = outputs.hidden_states[-1]  # [batch, seq, hidden]

        # STEP 2: Integrate associative memory
        hidden_states, _ = self.associative_memory(hidden_states)

        # STEP 3: Pool by attending to last valid token
        if attention_mask is not None:
            seq_lens = torch.eq(attention_mask, 1).int().sum(-1) - 1
            batch_idx = torch.arange(input_ids.shape[0], device=input_ids.device)
            pooled = hidden_states[batch_idx, seq_lens, :]
        else:
            pooled = hidden_states[:, -1, :]

        # STEP 4: Classify
        return self.classifier(pooled)

    def trigger_nested_consolidation(self):
        """Placeholder for consolidation (future: fast/medium/slow weight consolidation)"""
        pass


# =====================================================================
# PART 2: DATA LOADING
# =====================================================================

dataset = load_dataset('SetFit/ag_news', split='train')

def create_task(dataset, class_labels, num_samples=SAMPLES):
    filtered = dataset.filter(lambda x: x['label'] in class_labels)
    sampled = filtered.select(range(min(num_samples, len(filtered))))
    texts = [item['text'] for item in sampled]
    labels = [item['label'] % 2 for item in sampled]
    return texts, labels

task_a_texts, task_a_labels = create_task(dataset, [0, 1], SAMPLES)
task_b_texts, task_b_labels = create_task(dataset, [2, 3], SAMPLES)

print(f"Task A: {len(task_a_texts)} samples (World vs Sports)")
print(f"Task B: {len(task_b_texts)} samples (Business vs Sci/Tech)")


# =====================================================================
# PART 3: SHARED CLASSIFIER MODEL
# =====================================================================

class SharedClassifierModel(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model
        self.classifier = nn.Linear(HIDDEN_SIZE, 2, dtype=torch.bfloat16)

    def forward(self, input_ids, attention_mask=None):
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        hidden_states = outputs.hidden_states[-1]

        if attention_mask is not None:
            seq_lens = torch.eq(attention_mask, 1).int().sum(-1) - 1
            batch_idx = torch.arange(input_ids.shape[0], device=input_ids.device)
            last_hidden = hidden_states[batch_idx, seq_lens, :]
        else:
            last_hidden = hidden_states[:, -1, :]

        return self.classifier(last_hidden)


# =====================================================================
# PART 4: TOPOLOGICAL MODEL
# =====================================================================

class TopologicalModel(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model
        dev = next(base_model.parameters()).device
        self.classifier_A = nn.Linear(HIDDEN_SIZE, 2, dtype=torch.bfloat16).to(dev)
        self.classifier_B = nn.Linear(HIDDEN_SIZE, 2, dtype=torch.bfloat16).to(dev)
        self.current_task = 'A'

    def forward(self, input_ids, attention_mask=None):
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        hidden_states = outputs.hidden_states[-1]

        if attention_mask is not None:
            seq_lens = torch.eq(attention_mask, 1).int().sum(-1) - 1
            batch_idx = torch.arange(input_ids.shape[0], device=input_ids.device)
            last_hidden = hidden_states[batch_idx, seq_lens, :]
        else:
            last_hidden = hidden_states[:, -1, :]

        head = getattr(self, f'classifier_{self.current_task}')
        return head(last_hidden)

    def switch_task(self, task: str):
        assert task in ('A', 'B')
        self.current_task = task

    def freeze_previous_head(self):
        self.classifier_A.requires_grad_(False)


# =====================================================================
# PART 5: TOPOLOGICAL GOVERNOR
# =====================================================================

class TopologicalGovernor:
    def __init__(self, embed_layer: nn.Embedding, prime_limit: int = PRIME_LIMIT):
        self.embed_layer = embed_layer
        vocab_size = embed_layer.weight.shape[0]

        sieve = [True] * (prime_limit + 1)
        sieve[0] = sieve[1] = False
        for i in range(2, int(prime_limit ** 0.5) + 1):
            if sieve[i]:
                for j in range(i * i, prime_limit + 1, i):
                    sieve[j] = False
        primes = [i for i in range(2, prime_limit + 1) if sieve[i]]
        self.anchor_indices = [p for p in primes if p < vocab_size]
        self.snapshot = {}
        self.safety_constant = 1.0 - np.prod([1.0 - (p ** -0.5) for p in self.anchor_indices])

        print(f"  [GOVERNOR] Anchoring {len(self.anchor_indices)} prime coords: {self.anchor_indices}")
        print(f"  [GOVERNOR] Safety Constant Λ: {self.safety_constant:.10f}")

    def take_snapshot(self):
        self.snapshot = {
            idx: self.embed_layer.weight[idx].detach().clone().float()
            for idx in self.anchor_indices
        }

    @torch.no_grad()
    def zero_anchor_gradients(self):
        if self.embed_layer.weight.grad is not None:
            for idx in self.anchor_indices:
                self.embed_layer.weight.grad[idx].zero_()

    @torch.no_grad()
    def enforce_anchors(self):
        if not self.snapshot:
            return
        dtype = self.embed_layer.weight.dtype
        for idx, cached in self.snapshot.items():
            self.embed_layer.weight[idx].copy_(cached.to(dtype=dtype))

    def verify_integrity(self, atol: float = 1e-5) -> bool:
        if not self.snapshot:
            return True
        return all(
            torch.allclose(self.embed_layer.weight[idx].float(), cached, atol=atol)
            for idx, cached in self.snapshot.items()
        )


# =====================================================================
# PART 6: EWC
# =====================================================================

class EWC:
    def __init__(self, model, fisher_samples=200):
        self.model = model
        self.fisher_samples = fisher_samples
        self.params = {n: p for n, p in model.named_parameters() if p.requires_grad}
        self._means = {}
        self._precision_matrices = {}

    def update_fisher(self, dataloader):
        for n, p in self.params.items():
            self._precision_matrices[n] = torch.zeros_like(p).to(device)

        self.model.eval()
        for batch in dataloader:
            self.model.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            logits = self.model(input_ids, attention_mask)
            loss = F.cross_entropy(logits, labels)
            loss.backward()

            for n, p in self.params.items():
                if p.grad is not None:
                    self._precision_matrices[n] += p.grad ** 2 / len(dataloader)

        self.model.train()

    def update_means(self):
        for n, p in self.params.items():
            self._means[n] = p.clone().detach()

    def penalty(self, lambda_ewc=5000):
        loss = 0
        for n, p in self.params.items():
            if n in self._precision_matrices:
                _loss = self._precision_matrices[n] * (p - self._means[n]) ** 2
                loss += _loss.sum()
        return lambda_ewc * loss


# =====================================================================
# PART 7: EXPERIENCE REPLAY
# =====================================================================

class ExperienceReplay:
    def __init__(self, buffer_size=200):
        self.buffer_size = buffer_size
        self.buffer = []

    def add(self, texts, labels):
        self.buffer.extend(list(zip(texts, labels)))
        if len(self.buffer) > self.buffer_size:
            self.buffer = self.buffer[-self.buffer_size:]

    def sample(self, batch_size=16):
        if len(self.buffer) < batch_size:
            return [], []
        indices = np.random.choice(len(self.buffer), batch_size, replace=False)
        texts, labels = zip(*[self.buffer[i] for i in indices])
        return list(texts), list(labels)


# =====================================================================
# PART 8: EVALUATION
# =====================================================================

def evaluate_model(model, tokenizer, texts, labels, batch_size=16):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i+batch_size]
            batch_labels = labels[i:i+batch_size]
            tokens = tokenizer(batch_texts, return_tensors="pt", padding=True,
                             truncation=True, max_length=64).to(device)

            # Check if model is Full HOPE
            if hasattr(model, 'associative_memory'):
                logits = model(tokens.input_ids, tokens.attention_mask, update_fast=False)
            else:
                logits = model(tokens.input_ids, tokens.attention_mask)

            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(batch_labels)

    return accuracy_score(all_labels, all_preds)


# =====================================================================
# PART 9: TRAINING FUNCTION
# =====================================================================

def train_model(method='baseline'):
    print(f"\n{'='*60}")
    print(f"Method: {method.upper()}")
    print(f"{'='*60}")

    all_results = []

    for run_id in range(NUM_RUNS):
        lr_embed, lr_cls = LR_GRID[run_id]
        print(f"\n  Run {run_id+1}/{NUM_RUNS} | lr_embed={lr_embed:.0e}, lr_cls={lr_cls:.0e}")

        base_model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME, trust_remote_code=True, torch_dtype=torch.bfloat16
        ).to(device)

        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
        tokenizer.pad_token = tokenizer.eos_token

        for param in base_model.parameters():
            param.requires_grad = False

        # Initialize variables before model creation
        ewc = None
        replay = None
        governor = None
        embed_layer = None

        # Create model
        if method == 'topological':
            model = TopologicalModel(base_model).to(device)
        elif method == 'full_hope':
            model = FullHOPEModel(base_model).to(device)
        else:
            model = SharedClassifierModel(base_model).to(device)

        # Setup optimizer
        if method == 'topological':
            if hasattr(base_model, "transformer") and hasattr(base_model.transformer, "wte"):
                embed_layer = base_model.transformer.wte
            else:
                for module in base_model.modules():
                    if hasattr(module, "weight") and module.weight.shape[0] > 100000:
                        embed_layer = module
                        break
            embed_layer.weight.requires_grad = True
            governor = TopologicalGovernor(embed_layer)

            optimizer = bnb.optim.AdamW8bit([
                {'params': embed_layer.weight, 'lr': lr_embed},
                {'params': model.classifier_A.parameters(), 'lr': lr_cls}
            ])
        elif method == 'full_hope':
            model.classifier.requires_grad_(True)
            optimizer = bnb.optim.AdamW8bit(model.classifier.parameters(), lr=lr_cls)
        else:
            model.classifier.requires_grad_(True)
            optimizer = bnb.optim.AdamW8bit(model.classifier.parameters(), lr=lr_cls)

        # ===== TASK A TRAINING =====
        if method == 'topological':
            model.switch_task('A')
        model.train()

        for epoch in range(EPOCHS):
            for i in range(0, len(task_a_texts), BATCH_SIZE):
                batch_texts = task_a_texts[i:i+BATCH_SIZE]
                batch_labels = torch.tensor(task_a_labels[i:i+BATCH_SIZE]).to(device)

                tokens = tokenizer(batch_texts, return_tensors="pt", padding=True,
                                 truncation=True, max_length=64).to(device)
                optimizer.zero_grad()

                if method == 'full_hope':
                    logits = model(tokens.input_ids, tokens.attention_mask, update_fast=True)
                else:
                    logits = model(tokens.input_ids, tokens.attention_mask)

                loss = F.cross_entropy(logits, batch_labels)
                loss.backward()

                if method == 'topological' and governor:
                    governor.zero_anchor_gradients()

                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

        # ===== POST-TASK A =====
        if method == 'topological' and governor:
            governor.take_snapshot()
            model.freeze_previous_head()
            optimizer = bnb.optim.AdamW8bit([
                {'params': embed_layer.weight, 'lr': lr_embed},
                {'params': model.classifier_B.parameters(), 'lr': lr_cls}
            ])

        elif method == 'ewc':
            ewc = EWC(model)
            dummy_dataloader = []
            for i in range(0, len(task_a_texts), 32):
                batch_texts = task_a_texts[i:i+32]
                batch_labels = torch.tensor(task_a_labels[i:i+32]).to(device)
                tokens = tokenizer(batch_texts, return_tensors="pt", padding=True,
                                 truncation=True, max_length=64).to(device)
                dummy_dataloader.append({
                    'input_ids': tokens.input_ids,
                    'attention_mask': tokens.attention_mask,
                    'labels': batch_labels
                })
            ewc.update_fisher(dummy_dataloader)
            ewc.update_means()

        elif method == 'replay':
            replay = ExperienceReplay(buffer_size=200)
            replay.add(task_a_texts, task_a_labels)

        # ===== EVALUATE TASK A =====
        acc_a_initial = evaluate_model(model, tokenizer, task_a_texts[:200], task_a_labels[:200])
        print(f"      Task A Initial: {acc_a_initial*100:.1f}%")

        # ===== TASK B TRAINING =====
        if method == 'topological':
            model.switch_task('B')
        model.train()

        for epoch in range(EPOCHS):
            for i in range(0, len(task_b_texts), BATCH_SIZE):
                batch_texts = task_b_texts[i:i+BATCH_SIZE]
                batch_labels = torch.tensor(task_b_labels[i:i+BATCH_SIZE]).to(device)

                if method == 'replay' and replay:
                    replay_texts, replay_labels = replay.sample(batch_size=8)
                    if replay_texts:
                        batch_texts = batch_texts[:8] + replay_texts
                        batch_labels = torch.cat([
                            batch_labels[:8],
                            torch.tensor(replay_labels).to(device)
                        ])

                tokens = tokenizer(batch_texts, return_tensors="pt", padding=True,
                                 truncation=True, max_length=64).to(device)
                optimizer.zero_grad()

                if method == 'full_hope':
                    logits = model(tokens.input_ids, tokens.attention_mask, update_fast=True)
                else:
                    logits = model(tokens.input_ids, tokens.attention_mask)

                loss = F.cross_entropy(logits, batch_labels)

                if method == 'ewc' and ewc:
                    loss += ewc.penalty(lambda_ewc=500)

                loss.backward()

                if method == 'topological' and governor:
                    governor.zero_anchor_gradients()

                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

                if method == 'topological' and governor:
                    governor.enforce_anchors()

                if method == 'full_hope':
                    if i % 50 == 0:
                        model.trigger_nested_consolidation()

        # ===== FINAL EVALUATION =====
        if method == 'topological':
            model.switch_task('A')
            acc_a_final = evaluate_model(model, tokenizer, task_a_texts[:200], task_a_labels[:200])
            model.switch_task('B')
            acc_b = evaluate_model(model, tokenizer, task_b_texts[:200], task_b_labels[:200])
        else:
            acc_a_final = evaluate_model(model, tokenizer, task_a_texts[:200], task_a_labels[:200])
            acc_b = evaluate_model(model, tokenizer, task_b_texts[:200], task_b_labels[:200])

        forgetting = (acc_a_initial - acc_a_final) * 100

        print(f"      Task A Final: {acc_a_final*100:.1f}%")
        print(f"      Task B Acc: {acc_b*100:.1f}%")
        print(f"      Forgetting: {forgetting:.1f}%")

        if method == 'topological' and governor:
            if governor.verify_integrity():
                print(f"      ✅ Anchors intact")

        all_results.append({
            'run_id': run_id,
            'lr_embed': lr_embed,
            'lr_cls': lr_cls,
            'acc_a_initial': acc_a_initial,
            'acc_a_final': acc_a_final,
            'forgetting': forgetting,
            'acc_b': acc_b
        })

        del model, base_model, tokenizer, optimizer
        if method == 'topological' and governor is not None:
            del governor
        elif method == 'ewc' and ewc is not None:
            del ewc
        elif method == 'replay' and replay is not None:
            del replay

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

    return {
        'results': all_results,
        'best_run': max(all_results, key=lambda x: x['acc_b']),
        'mean_forgetting': np.mean([r['forgetting'] for r in all_results]),
        'mean_acc_b': np.mean([r['acc_b'] for r in all_results]),
        'std_forgetting': np.std([r['forgetting'] for r in all_results]),
        'std_acc_b': np.std([r['acc_b'] for r in all_results]),
    }


# =====================================================================
# PART 10: RUN BENCHMARK
# =====================================================================
print("\n" + "=" * 80)
print("RUNNING BENCHMARK")
print("=" * 80)

methods = ['baseline', 'ewc', 'replay', 'full_hope', 'topological']
results = {}

for method in methods:
    results[method] = train_model(method=method)


# =====================================================================
# PART 11: RESULTS TABLE
# =====================================================================
print("\n" + "=" * 80)
print("FINAL RESULTS")
print("=" * 80)
print(f"{'Method':<15} {'Best Fgt':<12} {'Mean Fgt':<12} {'Best Task B':<12} {'Mean Task B':<12}")
print("-" * 70)

for method in methods:
    r = results[method]
    best = r['best_run']
    print(f"{method.upper():<15} {best['forgetting']:.1f}%{'':<6} "
          f"{r['mean_forgetting']:.1f}%{'':<6} "
          f"{best['acc_b']*100:.1f}%{'':<6} "
          f"{r['mean_acc_b']*100:.1f}%")

print("-" * 70)

# Ranking
print("\n" + "=" * 80)
print("RANKING (Best to Worst by Mean Forgetting):")
print("=" * 80)

sorted_methods = sorted(methods, key=lambda m: results[m]['mean_forgetting'])
for i, method in enumerate(sorted_methods, 1):
    r = results[method]
    print(f"{i}. {method.upper()}: {r['mean_forgetting']:.1f}% forgetting "
          f"(Best Task B: {r['best_run']['acc_b']*100:.1f}%)")

print("\n" + "=" * 80)
print("VERDICT:")
print("=" * 80)

topo_best = results['topological']['best_run']
hope_best = results['full_hope']['best_run']
replay_best = results['replay']['best_run']

print(f"🔵 TOPOLOGICAL AI:")
print(f"   • Forgetting: {topo_best['forgetting']:.1f}%")
print(f"   • Task B Accuracy: {topo_best['acc_b']*100:.1f}%")

print(f"\n🟣 FULL HOPE:")
print(f"   • Forgetting: {hope_best['forgetting']:.1f}%")
print(f"   • Task B Accuracy: {hope_best['acc_b']*100:.1f}%")

print(f"\n🟢 EXPERIENCE REPLAY:")
print(f"   • Forgetting: {replay_best['forgetting']:.1f}%")
print(f"   • Task B Accuracy: {replay_best['acc_b']*100:.1f}%")

if topo_best['forgetting'] < hope_best['forgetting']:
    improvement = hope_best['forgetting'] - topo_best['forgetting']
    print(f"\n✅ TOPOLOGICAL AI WINS!")
    print(f"   • {improvement:.1f}% less forgetting than Full HOPE")
elif hope_best['forgetting'] < topo_best['forgetting']:
    improvement = topo_best['forgetting'] - hope_best['forgetting']
    print(f"\n✅ FULL HOPE WINS!")
    print(f"   • {improvement:.1f}% less forgetting than Topological AI")
else:
    print(f"\n⚖️ TIE!")

print("=" * 80)

Device: cuda
BENCHMARK: BASELINE | EWC | REPLAY | FULL HOPE | TOPOLOGICAL


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Task A: 1500 samples (World vs Sports)
Task B: 1500 samples (Business vs Sci/Tech)

RUNNING BENCHMARK

Method: BASELINE

  Run 1/5 | lr_embed=5e-03, lr_cls=1e-03


[transformers] MXFP4 quantization requires the `kernels` package: `pip install kernels>=0.12.0`. We will default to dequantizing the model to bf16.


Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

...

  [GOVERNOR] Anchoring 6 prime coords: [2, 3, 5, 7, 11, 13]
  [GOVERNOR] Safety Constant Λ: 0.9785142874
      Task A Initial: 98.0%
      Task A Final: 93.5%
      Task B Acc: 98.5%
      Forgetting: 4.5%
      ✅ Anchors intact

  Run 4/5 | lr_embed=5e-03, lr_cls=5e-03


Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

  [GOVERNOR] Anchoring 6 prime coords: [2, 3, 5, 7, 11, 13]
  [GOVERNOR] Safety Constant Λ: 0.9785142874
      Task A Initial: 97.0%
      Task A Final: 94.5%
      Task B Acc: 98.0%
      Forgetting: 2.5%
      ✅ Anchors intact

  Run 5/5 | lr_embed=2e-03, lr_cls=1e-03


Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

  [GOVERNOR] Anchoring 6 prime coords: [2, 3, 5, 7, 11, 13]
  [GOVERNOR] Safety Constant Λ: 0.9785142874
      Task A Initial: 98.5%
      Task A Final: 98.5%
      Task B Acc: 99.5%
      Forgetting: 0.0%
      ✅ Anchors intact

FINAL RESULTS
Method          Best Fgt     Mean Fgt     Best Task B  Mean Task B 
----------------------------------------------------------------------
BASELINE        20.5%       22.0%       87.5%       84.3%
EWC             10.0%       8.3%       78.0%       68.4%
REPLAY          7.5%       9.1%       84.0%       76.6%
FULL_HOPE       18.5%       8.5%       83.0%       73.8%
TOPOLOGICAL     0.0%       1.9%       99.5%       96.8%
----------------------------------------------------------------------

RANKING (Best to Worst by Mean Forgetting):
1. TOPOLOGICAL: 1.9% forgetting (Best Task B: 99.5%)
2. EWC: 8.3% forgetting (Best Task B: 78.0%)
3. FULL_HOPE: 8.5% forgetting (Best Task B: 83.0%)
4. REPLAY: 9.1% forgetting (Best Task B: 84.0%)
5. BASELINE: 22.0% f